# AlphaTrain: 5-Block Surrogate Model for Fast Self-Play

Train a small 5b×128ch model (2.9M params, 4x faster inference) for
self-play generation. Enables static 1600 sims at the speed of dynamic.

- Trained on V6 data (5.4M states from 2R ep3 teacher)
- Pure policy distillation (val_weight=0)
- **From scratch** (different architecture, can't warm-start from 10-block)
- 15 epochs (small model, needs more training)

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v6_density_g098_ms200.pt.gz` — V6 tensor (on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Decompress V6 data (same policy targets, used for distillation)
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Decompressing V6 data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v6_density_g098_ms200.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Surrogate: 5b×128ch, from scratch, pure policy
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --num-blocks 5 --channels 128 \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.0 --rank-weight 0.0 \
    --epochs 15 --batch-size 24576 --lr 1e-3 --warmup-epochs 3 \
    --copy-to /content/drive/MyDrive/alphatrain/surrogate_5b128_best.pt \
    --save-dir /content/checkpoints/surrogate

In [ ]:
# Copy checkpoints to Drive
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/surrogate/epoch_*.pt')):
    dst = f'{DRIVE}/surrogate_5b128_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/surrogate/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/surrogate_5b128_{f}')